# Mass, dust, redshift, and AB photometry

This notebook repeats the operations bib applies to a BPASS SED. Each step
writes the equation, evaluates it, and checks the result against the function
the server calls. The SED here is a smooth stand-in, so the notebook does not
need a database. The same functions run on the real grids.

Order of operations, as in `GET /api/v1/photometry` and `GET /api/v1/populations/{id}/sed`:

1. Scale the stored \(L_\lambda\) from \(10^6\,M_\odot\) formed to the requested mass.
2. Extinguish that rest-frame spectrum (a foreground screen).
3. Redshift: stretch wavelength and convert to observed \(f_\lambda\).
4. Integrate the observed \(f_\lambda\) through the filter for an AB magnitude.

Dust is extinction, not a mixed star–dust attenuation, and the light removed
is not re-emitted. Calzetti et al. (2000) published their curve as an
attenuation law; bib still multiplies by \(10^{-0.4 A_\lambda}\), the same
screen as the other curves.


In [1]:
import numpy as np
import astropy.units as u
from astropy.cosmology import FlatLambdaCDM
from dust_extinction.parameter_averages import CCM89

from bib.science.dust import apply_dust
from bib.science.photometry import AB_ZERO_FNU, C_A_S, FilterCurve, ab_magnitude
from bib.science.redshift import observe
from bib.science.sed import L_SUN_ERG_S, PC_CM, flux_at_10pc, scale_by_mass

# Stand-in rest-frame SED: L_sun/Å per 1e6 Msun formed, inside the CCM89 window.
wl = np.linspace(1200.0, 25000.0, 4000)
flux = 2.0e4 * (wl / 5500.0) ** -1.2


## Formed mass

BPASS tabulates a population of \(10^6\,M_\odot\) formed. A different formed
mass is a constant factor on \(L_\lambda\):

\[
L_\lambda(M) = L_\lambda(10^6\,M_\odot)\times (M / 10^6\,M_\odot).
\]


In [2]:
mass = 3.2e7  # Msun
scaled = scale_by_mass(flux, mass)
manual_scaled = flux * (mass / 1.0e6)
assert np.allclose(scaled, manual_scaled)
print(f"scale_by_mass matches L_lambda * (M / 1e6) at {mass:.3e} Msun")


scale_by_mass matches L_lambda * (M / 1e6) at 3.200e+07 Msun


## Dust extinction

\[
A_V = R_V\,E(B-V), \qquad
F'_\lambda = F_\lambda\,10^{-0.4 A_\lambda}, \qquad
A_\lambda = (A_\lambda/A_V)\,A_V.
\]

CCM89 and Fitzpatrick (1999) come from the `dust_extinction` package
(`dust_extinction.parameter_averages`). The SMC bar curve is
`dust_extinction.averages.G03_SMCBar`. Calzetti et al. (2000) is the
polynomial below, evaluated in the server as \(k(\lambda)/R_V\) with
\(R_V = 4.05\). Outside a curve's wavelength window the server clamps
\(x = 1/\lambda\) to the edge of that window before evaluating the curve.


In [3]:
ebv, rv = 0.15, 3.1
dusted = apply_dust(wl, scaled, "ccm89", ebv, rv)

x = 1.0e4 / wl  # 1/micron
eps = 1e-6
x_clamped = np.clip(x, 0.3 + eps, 10.0 - eps)
a_over_av = np.asarray(CCM89(Rv=rv)(x_clamped / u.micron), dtype=float)
a_lambda = a_over_av * rv * ebv
manual_dust = scaled * 10.0 ** (-0.4 * a_lambda)
assert np.allclose(dusted.flux, manual_dust)
print(f"CCM89 screen: A_V = {dusted.a_v:.4f}, max |fractional residual| = {np.max(np.abs(dusted.flux / manual_dust - 1)):.1e}")

# Calzetti et al. (2000), valid ~0.12–2.2 µm. k(λ) as published, then A(λ)/A_V = k/R_V.
wl_c = np.linspace(1300.0, 21000.0, 1500)
base = np.full_like(wl_c, 1.0e3)
rv_c, ebv_c = 4.05, 0.2
got = apply_dust(wl_c, base, "calzetti00", ebv_c, None)
xx = 1.0e4 / wl_c
k = np.where(
    xx < 1.0 / 0.63,
    2.659 * (-1.857 + 1.040 * xx) + rv_c,
    2.659 * (-2.156 + 1.509 * xx - 0.198 * xx**2 + 0.011 * xx**3) + rv_c,
)
a_over_c = np.clip(k, 0.0, None) / rv_c
expect = base * 10.0 ** (-0.4 * a_over_c * rv_c * ebv_c)
assert np.allclose(got.flux, expect)
print(f"Calzetti (2000) polynomial matches apply_dust; R_V fixed at {got.rv}")


CCM89 screen: A_V = 0.4650, max |fractional residual| = 6.7e-16
Calzetti (2000) polynomial matches apply_dust; R_V fixed at 4.05


## Redshift

Rest-frame \(L_\lambda\) is a luminosity density. At redshift \(z\) the
observed wavelength and flux density are

\[
\lambda_\mathrm{obs} = \lambda_\mathrm{rest}(1+z), \qquad
f_\lambda(\lambda_\mathrm{obs})
  = \frac{L_\lambda(\lambda_\mathrm{rest})\,L_\odot}{4\pi D_L^2\,(1+z)}.
\]

\(L_\odot = 3.848\times 10^{26}\,\mathrm{W}\) (the BPASS manual value).
\(D_L\) is from flat ΛCDM with \(H_0 = 70\,\mathrm{km\,s^{-1}\,Mpc^{-1}}\)
and \(\Omega_m = 0.3\), computed with `astropy.cosmology.FlatLambdaCDM`.
The factor \((1+z)\) is the bandwidth stretch, so the bolometric flux is
\(L_\mathrm{bol}/(4\pi D_L^2)\). Dust has already been applied in the rest
frame. \(z = 0\) places the source at 10 pc, which is the absolute-magnitude
frame.

The SSP Explorer plots that same spectrum in \(L_\odot\,\mathrm{Å}^{-1}\)
so it can share an axis with the \(z = 0\) curve:

\[
L_\lambda^\mathrm{plot}
  = L_\lambda^\mathrm{rest}\,\frac{(10\,\mathrm{pc}/D_L)^2}{1+z}.
\]

That is \(f_\lambda\) converted back by the 10 pc area,
\(f_\lambda\,4\pi (10\,\mathrm{pc})^2 / L_\odot\).
The browser estimates \(D_L\) with a trapezoidal integral of \(1/E(z)\).
The API uses astropy. The two distances agree to a few parts in \(10^6\)
over the redshift range of the slider.


In [4]:
z = 0.2
cosmo = FlatLambdaCDM(H0=70.0, Om0=0.3)
wl_obs, flam = observe(wl, dusted.flux, z)
dl_cm = float(cosmo.luminosity_distance(z).to_value("cm"))
flam_manual = dusted.flux * L_SUN_ERG_S / (4.0 * np.pi * dl_cm**2 * (1.0 + z))
assert np.allclose(wl_obs, wl * (1.0 + z))
assert np.allclose(flam, flam_manual)

plot = dusted.flux * (10.0 / (dl_cm / PC_CM)) ** 2 / (1.0 + z)
plot_from_flam = flam * 4.0 * np.pi * (10.0 * PC_CM) ** 2 / L_SUN_ERG_S
assert np.allclose(plot, plot_from_flam)
print(f"z = {z}: API f_lambda matches L_lambda * Lsun / (4 pi D_L^2 (1+z))")
print(f"explorer plot level matches f_lambda * 4 pi (10 pc)^2 / Lsun")


def luminosity_distance_mpc_as_in_the_browser(z, h0=70.0, om0=0.3):
    # Trapezoidal D_L used by sed.js. z > 0.
    ol = 1.0 - om0
    dh = 299792.458 / h0  # Mpc
    n = max(48, int(np.ceil(z * 80)))
    dz = z / n

    def e(zi):
        return np.sqrt(om0 * (1.0 + zi) ** 3 + ol)

    integ = 0.5 / e(0.0)
    for i in range(1, n):
        integ += 1.0 / e(i * dz)
    integ += 0.5 / e(z)
    return (1.0 + z) * dh * integ * dz


print("D_L (Mpc): astropy vs the explorer trapezoid")
for zz in (0.1, 1.0, 2.0, 10.0):
    astropy_dl = float(cosmo.luminosity_distance(zz).to_value("Mpc"))
    browser_dl = luminosity_distance_mpc_as_in_the_browser(zz)
    print(f"  z = {zz:4.1f}   {astropy_dl:12.4f}   {browser_dl:12.4f}   rel {(browser_dl - astropy_dl) / astropy_dl:+.3e}")
    assert abs(browser_dl - astropy_dl) / astropy_dl < 1e-4


z = 0.2: API f_lambda matches L_lambda * Lsun / (4 pi D_L^2 (1+z))
explorer plot level matches f_lambda * 4 pi (10 pc)^2 / Lsun
D_L (Mpc): astropy vs the explorer trapezoid
  z =  0.1       460.2999       460.2999   rel -8.297e-08
  z =  1.0      6607.6576      6607.6710   rel +2.029e-06
  z =  2.0     15539.5862     15539.6356   rel +3.174e-06
  z = 10.0    103842.7459    103843.0178   rel +2.618e-06


## AB photometry

For a photon-counting detector (the bundled SVO filters, including NIRCam):

\[
m_\mathrm{AB} = -2.5\log_{10}
\frac{\int f_\lambda\,T(\lambda)\,\lambda\,d\lambda}
     {\int f_\lambda^{3631\,\mathrm{Jy}}\,T(\lambda)\,\lambda\,d\lambda}.
\]

The reference spectrum is \(f_\nu = 3631\,\mathrm{Jy}\), written as
\(f_\lambda = 3631\,\mathrm{Jy}\times c/\lambda^2\).
Transmission curves are scaled to a peak of 1. The integral is a trapezoidal
rule on the SED wavelength grid, with \(T(\lambda)\) interpolated onto that
grid. Energy-counting detectors omit the extra factor of \(\lambda\).

At \(z = 0\), \(f_\lambda\) is \(L_\lambda L_\odot / 4\pi (10\,\mathrm{pc})^2\).
At \(z > 0\), \(f_\lambda\) is the observer-frame value from the redshift
step, so the magnitude is apparent and includes the K-correction.


In [5]:
band = (wl > 4500.0) & (wl < 6500.0)
filt = FilterCurve(
    "demo.g",
    "demo g",
    wl[band],
    np.exp(-0.5 * ((wl[band] - 5500.0) / 250.0) ** 2),
    detector_type="photon",
)
result = ab_magnitude(wl, flux, filt)  # absolute: flux is still L_sun/Å

inside = (wl >= filt.support[0]) & (wl <= filt.support[1])
w = wl[inside]
f_lambda = flux_at_10pc(flux[inside])
transmission = np.interp(w, filt.wavelength, filt.transmission, left=0.0, right=0.0)
ab_f_lambda = AB_ZERO_FNU * C_A_S / w**2
numerator = np.trapezoid(f_lambda * transmission * w, w)
denominator = np.trapezoid(ab_f_lambda * transmission * w, w)
mag = -2.5 * np.log10(numerator / denominator)
assert abs(result.ab_mag - mag) < 1e-12
print(f"absolute AB = {result.ab_mag:.6f}  (library and the integral agree)")


absolute AB = -15.628599  (library and the integral agree)


## The four steps together

Photometry at redshift uses the extinguished, mass-scaled, observer-frame
\(f_\lambda\). Because every step before the integral is linear in flux,
changing the formed mass by a factor \(s\) changes the magnitude by
\(-2.5\log_{10} s\). Dust and redshift do not change that factor.


In [6]:
def apparent(formed_mass):
    rest = apply_dust(wl, scale_by_mass(flux, formed_mass), "ccm89", 0.1, 3.1).flux
    observed_wl, observed_f = observe(wl, rest, 0.15)
    return ab_magnitude(observed_wl, observed_f, filt, flux_unit="flam_cgs").ab_mag

m_1e6 = apparent(1.0e6)
m_1e7 = apparent(1.0e7)
expected = -2.5 * np.log10(1.0e7 / 1.0e6)
print(f"m(10^6 Msun) = {m_1e6:.4f}")
print(f"m(10^7 Msun) = {m_1e7:.4f}")
print(f"m(10^6) - m(10^7) = {m_1e6 - m_1e7:.4f} mag")
print(f"a factor of ten in mass changes the magnitude by {-expected:.4f}")
assert abs((m_1e6 - m_1e7) - (-expected)) < 1e-10


m(10^6 Msun) = 23.9769
m(10^7 Msun) = 21.4769
m(10^6) - m(10^7) = 2.5000 mag
a factor of ten in mass changes the magnitude by 2.5000
